# linear-affine-on-custom-tensor — faded example 1: Fill the matmul in the Linear forward

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `linear-affine-on-custom-tensor`. The last cell reports your progress on the `Backprop: Linear affine on custom Tensor` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Linear affine on custom Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`linear-affine-on-custom-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "linear-affine-on-custom-tensor"
DD_SUBTOPIC = "Backprop: Linear affine on custom Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The custom Linear forward is `out = x @ weight + bias`. The first step contracts the input feature axis: `(B, in) @ (in, out) -> (B, out)`. The bias of shape `(out,)` then broadcasts across the batch axis.

## Faded exercise 1

Complete `linear_forward(x, weight, bias)`. Everything is filled except the matmul array computation that produces the `(B, out)` intermediate. Fill in that one expression.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import numpy as np
from dataclasses import dataclass

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad
        self.recipe = recipe

def linear_forward(x, weight, bias):
    mm_arr = None  # TODO: fill in this step — read the prompt cell above
    mm = MiniTensor(mm_arr, requires_grad=(x.requires_grad or weight.requires_grad))
    mm.recipe = Recipe(np.matmul, (x.array, weight.array), {}, {0: x, 1: weight})
    out = MiniTensor(mm.array + bias.array,
                     requires_grad=(mm.requires_grad or bias.requires_grad))
    out.recipe = Recipe(np.add, (mm.array, bias.array), {}, {0: mm, 1: bias})
    return out


def _test():
    np.random.seed(7)
    x = MiniTensor(np.random.randn(4, 3))
    weight = MiniTensor(np.random.randn(3, 5), requires_grad=True)
    bias = MiniTensor(np.random.randn(5), requires_grad=True)
    out = linear_forward(x, weight, bias)
    # independent ground truth via explicit loops
    B, IN, OUT = 4, 3, 5
    expected = np.zeros((B, OUT))
    for b in range(B):
        for o in range(OUT):
            acc = 0.0
            for i in range(IN):
                acc += x.array[b, i] * weight.array[i, o]
            expected[b, o] = acc + bias.array[o]
    assert out.array.shape == (B, OUT)
    assert np.allclose(out.array, expected)
    assert out.requires_grad is True
    assert out.recipe.parents[1] is bias


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
from dataclasses import dataclass

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad
        self.recipe = recipe

def linear_forward(x, weight, bias):
    mm_arr = x.array @ weight.array
    mm = MiniTensor(mm_arr, requires_grad=(x.requires_grad or weight.requires_grad))
    mm.recipe = Recipe(np.matmul, (x.array, weight.array), {}, {0: x, 1: weight})
    out = MiniTensor(mm.array + bias.array,
                     requires_grad=(mm.requires_grad or bias.requires_grad))
    out.recipe = Recipe(np.add, (mm.array, bias.array), {}, {0: mm, 1: bias})
    return out
```
</details>